# 00 — Data snapshot

A look at the data before any analysis: the AACT tables we extracted, how they link, one trial followed through
every table, and the one-row-per-trial table the notebooks use.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC = ROOT / "data/raw", ROOT / "data/processed"
pd.set_option("display.max_colwidth", 80)
print((RAW / "SNAPSHOT.txt").read_text())

Extracted from AACT cloud on 2026-10-07
Cohort: interventional trials with start_date 2008-01-01 to 2020-12-31



## 1. Extracted tables

`scripts/02_extract.py` saves each AACT table to `data/raw/<table>.parquet`, restricted to the cohort. Every table
has `nct_id`, the trial's ClinicalTrials.gov ID. Tables with more than one row per trial hold lists, such as sites or
sponsors.

In [2]:
ABOUT = {
    "studies": "status, why_stopped, phase, dates, enrollment",
    "designs": "allocation, masking, intervention model, primary purpose",
    "eligibilities": "criteria text, age and sex limits, healthy volunteers",
    "calculated_values": "fields AACT derives (duration, number of facilities, ages in years)",
    "sponsors": "lead sponsor and collaborators",
    "conditions": "conditions as written by the sponsor",
    "browse_conditions": "MeSH terms for the conditions",
    "interventions": "drugs, devices, procedures",
    "countries": "countries with sites (removed = dropped later)",
    "facilities": "site city, state and country",
}
rows = []
for name, about in ABOUT.items():
    path = RAW / f"{name}.parquet"
    meta = pq.ParquetFile(path).metadata
    trials = pd.read_parquet(path, columns=["nct_id"])["nct_id"].nunique()
    rows.append({"table": name, "rows": meta.num_rows, "columns": meta.num_columns, "trials": trials,
                 "rows_per_trial": round(meta.num_rows / trials, 1), "contents": about})
pd.DataFrame(rows).set_index("table").style.format({"rows": "{:,}", "trials": "{:,}", "rows_per_trial": "{:.1f}"})

,rows,columns,trials,rows_per_trial,contents
table,,,,,
studies,"231,776",71,"231,776",1.0,"status, why_stopped, phase, dates, enrollment"
designs,"231,664",14,"231,664",1.0,"allocation, masking, intervention model, primary purpose"
eligibilities,"231,776",14,"231,776",1.0,"criteria text, age and sex limits, healthy volunteers"
calculated_values,"231,776",19,"231,776",1.0,"fields AACT derives (duration, number of facilities, ages in years)"
sponsors,"371,444",5,"231,776",1.6,lead sponsor and collaborators
conditions,"387,631",4,"231,776",1.7,conditions as written by the sponsor
browse_conditions,"1,637,527",5,"183,542",8.9,MeSH terms for the conditions
interventions,"463,909",5,"231,776",2.0,"drugs, devices, procedures"
countries,"352,572",4,"215,833",1.6,countries with sites (removed = dropped later)


## 2. One trial across the tables

All rows for one trial: NCT01453088, a Phase 2 trial terminated for lack of accrual (`label = 1`).

In [3]:
EXAMPLE = "NCT01453088"

def rows_for(table, columns=None):
    return pd.read_parquet(RAW / f"{table}.parquet", columns=columns, filters=[("nct_id", "==", EXAMPLE)])

rows_for("studies", ["nct_id", "brief_title", "overall_status", "why_stopped", "phase", "start_date",
                     "completion_date", "enrollment", "enrollment_type", "source_class"]).T

,0
nct_id,NCT01453088
brief_title,Auto Transplant High Dose Melphalan vs High Dose Melphalan+Bortezomib in Pts...
overall_status,TERMINATED
why_stopped,Lack of Accrual
phase,PHASE2
start_date,2010-06-24
completion_date,2022-05-01
enrollment,63.0
enrollment_type,ACTUAL
source_class,OTHER


In [4]:
display(rows_for("designs").drop(columns="id").T)
elig = rows_for("eligibilities")
display(elig.drop(columns=["id", "criteria"]).T)
print(elig["criteria"].iloc[0])

,0
nct_id,NCT01453088
allocation,RANDOMIZED
intervention_model,PARALLEL
observational_model,None
primary_purpose,TREATMENT
time_perspective,None
masking,NONE
masking_description,NaN
intervention_model_description,NaN
subject_masked,None


,0
nct_id,NCT01453088
sampling_method,None
gender,ALL
minimum_age,60 Years
maximum_age,NaN
healthy_volunteers,False
population,None
gender_description,NaN
gender_based,None
adult,True


Inclusion Criteria:

* Confirmed diagnosis of multiple myeloma less than 12 months since initiation of systemic therapy
* Age ≥60 years at time of transplantation
* KPS 70-100%
* Recovery from complications of prior therapy

Exclusion Criteria:

* Diagnosis other than multiple myeloma
* Chemotherapy or radiotherapy within 8 days of initiating treatment in this study
* Prior dose-intense therapy within 56 days of initiating treatment in this study
* Uncontrolled bacterial, viral, fungal or parasitic infections
* Uncontrolled CNS metastases
* Known amyloid deposition in heart
* Organ dysfunction
* LVEF \<40% or cardiac failure not responsive to therapy
* FVC, FEV1 or DLCO \< 40% of predicted and/or receiving supplementary continuous oxygen
* Evidence of hepatic synthetic dysfunction or total bilirubin \> 2x or AST \> 3x ULN
* Measured creatinine \< 20ml/min
* Sensory peripheral neuropathy grade 4 within 14 days of enrollment
* Karnofsky score \< 70%
* Life expectancy limited by other co-

In [5]:
for table in ["sponsors", "conditions", "browse_conditions", "interventions", "countries", "facilities"]:
    print(table)
    display(rows_for(table).drop(columns="id", errors="ignore"))

sponsors


,nct_id,agency_class,lead_or_collaborator,name
0,NCT01453088,OTHER,lead,Hackensack Meridian Health


conditions


,nct_id,name,downcase_name
0,NCT01453088,Multiple Myeloma,multiple myeloma


browse_conditions


,nct_id,mesh_term,downcase_mesh_term,mesh_type
0,NCT01453088,"Neoplasms, Plasma Cell","neoplasms, plasma cell",mesh-ancestor
1,NCT01453088,Neoplasms by Histologic Type,neoplasms by histologic type,mesh-ancestor
2,NCT01453088,Neoplasms,neoplasms,mesh-ancestor
3,NCT01453088,Hemostatic Disorders,hemostatic disorders,mesh-ancestor
4,NCT01453088,Vascular Diseases,vascular diseases,mesh-ancestor
5,NCT01453088,Cardiovascular Diseases,cardiovascular diseases,mesh-ancestor
6,NCT01453088,Paraproteinemias,paraproteinemias,mesh-ancestor
7,NCT01453088,Blood Protein Disorders,blood protein disorders,mesh-ancestor
8,NCT01453088,Hematologic Diseases,hematologic diseases,mesh-ancestor
9,NCT01453088,Hemic and Lymphatic Diseases,hemic and lymphatic diseases,mesh-ancestor


... (2 outputs left out)

countries


,nct_id,name,removed
0,NCT01453088,United States,False


facilities


,nct_id,city,state,country
0,NCT01453088,Washington D.C.,District of Columbia,United States
1,NCT01453088,Hackensack,New Jersey,United States


## 3. The analysis table

`scripts/03_build_dataset.py` combines these into `data/processed/trials.parquet`, one row per trial, and adds the
outcome: `label = 1` for trials terminated for low enrollment, `0` for completed trials, and empty for everything
else. Some columns describe the trial's final state rather than its start (`enrollment`, `completion_date`,
`n_facilities`, `n_countries`), so they are not used as model features; see the leakage warning in the README.

In [6]:
trials = pd.read_parquet(PROC / "trials.parquet")
print(f"{trials.shape[0]:,} trials x {trials.shape[1]} columns")
trials[trials["nct_id"] == EXAMPLE].drop(columns="criteria").T

231,776 trials x 36 columns


,169236
nct_id,NCT01453088
overall_status,TERMINATED
why_stopped,Lack of Accrual
phase,PHASE2
start_date,2010-06-24 00:00:00
completion_date,2022-05-01
enrollment,63.0
enrollment_type,ACTUAL
number_of_arms,2.0
has_dmc,True


In [7]:
trials.drop(columns=["criteria", "why_stopped"]).sample(5, random_state=0)

,nct_id,overall_status,phase,start_date,completion_date,enrollment,enrollment_type,number_of_arms,has_dmc,is_fda_regulated_drug,...,healthy_volunteers,adult,child,older_adult,criteria_n_chars,n_conditions,n_interventions,n_countries,n_facilities,us_site
70643,NCT01603550,COMPLETED,NA,2012-06-30,2015-06-30,31.0,ACTUAL,2.0,False,None,...,True,True,False,False,581.0,2,2,1.0,1.0,True
93381,NCT04484610,COMPLETED,PHASE4,2020-09-08,2022-03-31,205.0,ACTUAL,2.0,False,False,...,False,True,True,True,238.0,2,1,1.0,1.0,False
113730,NCT01814579,UNKNOWN,NA,2012-07-31,2013-08-31,80.0,ESTIMATED,2.0,False,None,...,False,True,False,True,258.0,1,2,1.0,2.0,True
27564,NCT01153724,COMPLETED,PHASE1,2010-05-31,None,35.0,ACTUAL,2.0,None,None,...,True,True,False,False,55.0,2,2,1.0,1.0,False
214644,NCT00622986,UNKNOWN,PHASE4,2008-02-29,2009-02-28,288.0,ESTIMATED,4.0,False,None,...,False,True,False,False,1260.0,1,4,1.0,13.0,False


In [8]:
label = trials["label"].map({1: "1", 0: "0"}).fillna("not labeled")
pd.crosstab(trials["overall_status"], label, margins=True, margins_name="total")

label,0,1,not labeled,total
overall_status,,,,
ACTIVE_NOT_RECRUITING,0,0,4982,4982
COMPLETED,156707,0,0,156707
ENROLLING_BY_INVITATION,0,0,215,215
NOT_YET_RECRUITING,0,0,66,66
RECRUITING,0,0,3017,3017
SUSPENDED,0,0,647,647
TERMINATED,0,7886,12153,20039
UNKNOWN,0,0,37943,37943
WITHDRAWN,0,0,8160,8160


## 4. Files built from it

Later notebooks save their outputs to `data/processed/` and `data/labels/`. Files that haven't been built on this
machine are listed with the notebook or script that makes them.

In [9]:
BUILT = {
    "data/processed/eligibility_features.parquet": "03_eligibility_features",
    "data/processed/disease_area.parquet": "04_disease_area",
    "data/processed/baseline_predictions.parquet": "05_baseline_model",
    "data/labels/stop_reasons/sample.csv": "scripts/stop_reason_sheets.py",
}
for path, source in BUILT.items():
    p = ROOT / path
    if not p.exists():
        print(f"{path}: not built yet (run {source})\n")
        continue
    df = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
    print(f"{path}: {len(df):,} rows x {df.shape[1]} columns ({source})")
    display(df.head(3))

data/processed/eligibility_features.parquet: not built yet (run 03_eligibility_features)

data/processed/disease_area.parquet: not built yet (run 04_disease_area)

data/processed/baseline_predictions.parquet: not built yet (run 05_baseline_model)

data/labels/stop_reasons/sample.csv: 500 rows x 12 columns (scripts/stop_reason_sheets.py)


,item_id,n_trials,text,v0,n_v0_matches,stratum,N_h,n_h,weight,role,assigned_to,split
0,SR001,1,Obtained results were not good. Protocol was proved ineffective.,other,0,other,4011,120,33.425000,calibration,all,dev
1,SR002,1,Significant difference at interim analysis,efficacy,1,efficacy,751,35,21.457143,calibration,all,dev
2,SR003,1,"with 2/3 of the data collected, we had to terminate the study because of COV...",covid,1,covid,621,35,17.742857,calibration,all,dev
